# Amazon ML Challenge 2026 — Google Colab GPU Pipeline

This notebook resumes and runs the complete end-to-end Entity Resolution pipeline on Google Colab with GPU acceleration.

### Pipeline Steps:
1. **Environment Setup & Hardware Check** (GPU verification, dependencies, Git LFS)
2. **Clone & Pull Repository with Git LFS** (Includes pre-indexed blocking checkpoints and normalized datasets)
3. **Run Pipeline**:
   - **Stage 1 (Blocking)**: Resumes automatically from the 343k+ S1 checkpoint using cached 2.3GB index.
   - **Stage 2a (Data Prep)**: Prepares contrastive pairs and evaluation splits.
   - **Stage 2b (BGE-M3 Fine-Tuning)**: LoRA training with GradCache on Colab GPU.
   - **Stage 2c (Dense Features)**: GPU-accelerated cosine similarity extraction.
   - **Stage 3 (GBM Scoring)**: Grouped-OOF XGBoost ranking with monotonic constraints.
   - **Stage 4 & 5 (Clustering & Submission)**: Generates and validates `submission.csv`.

## 1. Verify GPU Hardware

In [ ]:
!nvidia-smi

## 2. Clone Repository & Pull Git LFS Artifacts
Replace the repository URL or enter your GitHub credentials if private.

In [ ]:
!apt-get install -y git-lfs
!git lfs install

# Clone the repository (if not already cloned)
import os
if not os.path.exists("AmazonMLChallenge2026"):
    !git clone https://github.com/priyanshu-ogdev/AmazonMLChallenge2026.git
    %cd AmazonMLChallenge2026
else:
    %cd AmazonMLChallenge2026
    !git pull origin main

# Pull all Git LFS tracked files (datasets, 2.3GB index, candidate pairs)
!git lfs pull

## 3. Install Required Dependencies

In [ ]:
!pip install -r code/business_entity_resolution/requirements.txt
!chmod +x scripts/*.sh

## 4. Run Pre-flight Environment & Smoke Tests

In [ ]:
!bash scripts/00_verify_environment.sh --python-path python3

## 5. Execute Pipeline

Choose between:
- **Option A (Full GPU Pipeline)**: Trains BGE-M3 LoRA adapter, extracts dense embeddings with GPU, and trains XGBoost.
- **Option B (Fast Baseline)**: Skips dense embedding extraction for an ultra-fast baseline submission.

In [ ]:
# OPTION A: Full GPU Pipeline (Recommended on GPU runtime)
!bash scripts/run_all_phases.sh --run-mode Full --python-path python3

In [ ]:
# OPTION B: Fast Lexical Baseline (If you want a quick submission first)
# !bash scripts/run_all_phases.sh --run-mode Full --skip-gpu --baseline-only --python-path python3

## 6. Validate & Download Submission

In [ ]:
!bash scripts/05_validate_submission.sh --python-path python3

from google.colab import files
sub_path = "output/phase4_predictions/submission.csv"
if os.path.exists(sub_path):
    files.download(sub_path)
    print("Downloaded submission.csv!")
else:
    print(f"File not found: {sub_path}")